# Baseline — ce qu'un modèle de lettres tout fait écrit de nos tours

Aucun entraînement. C'est le chiffre contre lequel l'affinage se jugera : combien
d'hésitations un modèle entraîné sur des livres lus rend sur de la vraie parole
d'apprenant.

Réglages Kaggle : accélérateur **GPU T4** (ou aucun, c'est court), Internet **on**,
persistance **Files only**. Le dataset des prises se monte en lecture seule sous
`/kaggle/input/`.

Ce qui rentre à la maison : `baseline.json`, quelques kilo-octets.

In [ ]:
MODEL = "facebook/wav2vec2-large-960h-lv60-self"

In [ ]:
import json, pathlib

import soundfile
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained(MODEL)
model = Wav2Vec2ForCTC.from_pretrained(MODEL).eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"{MODEL}: {sum(p.numel() for p in model.parameters()):,} parameters on {device}")
print("vocabulary:", "".join(sorted(processor.tokenizer.get_vocab())))

In [ ]:
takes = sorted(pathlib.Path("/kaggle/input").rglob("*.wav"))
if not takes:
    raise SystemExit("no wav under /kaggle/input -- is the dataset attached?")
print(len(takes), "takes")

rows = []
for wav in takes:
    audio, rate = soundfile.read(wav)
    if rate != 16000:
        raise SystemExit(f"{wav}: {rate} Hz, expected 16000")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    heard = processor(audio, sampling_rate=rate, return_tensors="pt")
    with torch.no_grad():
        logits = model(heard.input_values.to(device),
                       attention_mask=heard.attention_mask.to(device)).logits
    text = processor.batch_decode(logits.argmax(dim=-1))[0]
    rows.append({"take": str(wav.relative_to("/kaggle/input")),
                 "seconds": round(len(audio) / rate, 2),
                 "text": text})
    print(f"{wav.name} ({rows[-1]['seconds']}s): {text}")

In [ ]:
out = pathlib.Path("/kaggle/working/baseline.json")
out.write_text(json.dumps({"model": MODEL, "takes": rows}, ensure_ascii=False, indent=2),
               encoding="utf-8")
print(out, out.stat().st_size, "bytes")